In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
data = pd.read_csv('/kaggle/input/fnnds-selected/fnnds_selected_columns_dataset.csv')

In [3]:
# Ensure the columns are clean
data.columns = [col.strip().replace("\n", " ").replace(" ", "_").lower() for col in data.columns]

In [4]:
# Define features and target columns
features = ['main_food_description']
targets = [
    'energy_(kcal)', 'protein_(g)', 'carbohydrate_(g)', 
    'sugars,_total_(g)', 'fiber,_total_dietary_(g)', 'total_fat_(g)', 
    'fatty_acids,_total_saturated_(g)',
    'fatty_acids,_total_monounsaturated_(g)',
    'fatty_acids,_total_polyunsaturated_(g)', 'cholesterol_(mg)'
]

In [5]:
X = data[features]
y = data[targets]

In [6]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
preprocessor = ColumnTransformer([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000), 'main_food_description')
])

In [8]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0, solver='lsqr', random_state=42))
])


In [9]:
# Train
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('tfidf',
                                                  TfidfVectorizer(max_features=5000,
                                                                  ngram_range=(1,
                                                                               2)),
                                                  'main_food_description')])),
                ('regressor', Ridge(random_state=42, solver='lsqr'))])

In [13]:
y_pred = pipeline.predict(X_test)
for i, target in enumerate(targets):
    mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"\nNutrient: {target}")
    print(f"  MSE: {mse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.2f}")


Nutrient: energy_(kcal)
  MSE: 4555.13
  MAE: 40.70
  R²: 0.77

Nutrient: protein_(g)
  MSE: 13.01
  MAE: 2.22
  R²: 0.78

Nutrient: carbohydrate_(g)
  MSE: 94.89
  MAE: 5.93
  R²: 0.78

Nutrient: sugars,_total_(g)
  MSE: 54.01
  MAE: 3.55
  R²: 0.59

Nutrient: fiber,_total_dietary_(g)
  MSE: 2.54
  MAE: 0.69
  R²: 0.52

Nutrient: total_fat_(g)
  MSE: 46.78
  MAE: 3.64
  R²: 0.63

Nutrient: fatty_acids,_total_saturated_(g)
  MSE: 9.14
  MAE: 1.39
  R²: 0.51

Nutrient: fatty_acids,_total_monounsaturated_(g)
  MSE: 7.13
  MAE: 1.42
  R²: 0.68

Nutrient: fatty_acids,_total_polyunsaturated_(g)
  MSE: 5.30
  MAE: 1.16
  R²: 0.57

Nutrient: cholesterol_(mg)
  MSE: 755.92
  MAE: 14.73
  R²: 0.80


In [14]:
def predict_nutrients(food_description, quantity_in_grams):
    input_data = pd.DataFrame({
        'main_food_description': [food_description]
    })

    # Predict nutrient values per 100g
    nutrient_values_100g = pipeline.predict(input_data)[0]

    # Scale predictions for the given quantity
    nutrient_values_scaled = (nutrient_values_100g * quantity_in_grams) / 100

    # Create a dictionary for the results
    nutrients = {
        'energy_(kcal)': nutrient_values_scaled[0],
        'protein_(g)': nutrient_values_scaled[1],
        'carbohydrate_(g)': nutrient_values_scaled[2],
        'sugars_total_(g)': nutrient_values_scaled[3],
        'fiber_total_dietary_(g)': nutrient_values_scaled[4],
        'total_fat_(g)': nutrient_values_scaled[5],
        'fatty_acids,_total_saturated_(g)': nutrient_values_scaled[6],
        'fatty_acids,_total_monounsaturated_(g)': nutrient_values_scaled[7],
        'fatty_acids,_total_polyunsaturated_(g)': nutrient_values_scaled[8],
        'cholesterol_(mg)': nutrient_values_scaled[9]
    }

    return nutrients

In [15]:
# Test cases
test_cases = [
    {"food": "Milk, whole", "quantity": 250},
    {"food": "Bread, whole wheat", "quantity": 100},
    {"food": "Apple, raw", "quantity": 150},
    {"food": "Cheese, cheddar", "quantity": 50},
    {"food": "Chicken, roasted", "quantity": 200},
    {"food": "Rice, white, cooked", "quantity": 180},
    {"food": "Broccoli, steamed", "quantity": 100},
    {"food": "Egg, boiled", "quantity": 70},
    {"food": "Butter, salted", "quantity": 20},
    {"food": "Banana, raw", "quantity": 120}
]

In [16]:
for case in test_cases:
    food = case["food"]
    quantity = case["quantity"]
    predicted_nutrients = predict_nutrients(food, quantity)
    
    print(f"\nPredicted nutrients for {quantity}g of {food}:")
    for nutrient, value in predicted_nutrients.items():
        print(f"{nutrient}: {value:.2f}")


Predicted nutrients for 250g of Milk, whole:
energy_(kcal): 211.14
protein_(g): 11.42
carbohydrate_(g): 17.15
sugars_total_(g): 12.80
fiber_total_dietary_(g): 0.76
total_fat_(g): 10.72
fatty_acids,_total_saturated_(g): 4.76
fatty_acids,_total_monounsaturated_(g): 3.33
fatty_acids,_total_polyunsaturated_(g): 1.32
cholesterol_(mg): 77.83

Predicted nutrients for 100g of Bread, whole wheat:
energy_(kcal): 277.74
protein_(g): 11.13
carbohydrate_(g): 47.14
sugars_total_(g): 6.45
fiber_total_dietary_(g): 7.26
total_fat_(g): 5.28
fatty_acids,_total_saturated_(g): 1.37
fatty_acids,_total_monounsaturated_(g): 1.26
fatty_acids,_total_polyunsaturated_(g): 1.83
cholesterol_(mg): 9.17

Predicted nutrients for 150g of Apple, raw:
energy_(kcal): 45.59
protein_(g): -1.98
carbohydrate_(g): 19.62
sugars_total_(g): 15.51
fiber_total_dietary_(g): 3.66
total_fat_(g): -1.59
fatty_acids,_total_saturated_(g): -0.27
fatty_acids,_total_monounsaturated_(g): -0.85
fatty_acids,_total_polyunsaturated_(g): -0.55
ch